In [8]:
import pandas as pd
import os

def validate_and_standardize(path, output_dir="validation_standardization_output"):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(path)

    df.columns = df.columns.str.lower().str.replace(" ", "_")

    missing_report = df.isnull().sum().reset_index()
    missing_report.columns = ["column_name", "missing_values"]
    missing_report.to_csv(
        f"{output_dir}/missing_values_report.csv",
        index=False
    )

    df = df.drop_duplicates()

    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.lower().str.strip()

    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

    for col in num_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val != min_val:
            df[col] = (df[col] - min_val) / (max_val - min_val)

    summary = pd.DataFrame({
        "total_rows": [df.shape[0]],
        "total_columns": [df.shape[1]],
        "remaining_missing_values": [df.isnull().sum().sum()],
        "standardization": ["column names, text lowercase, min-max numeric"]
    })

    summary.to_csv(
        f"{output_dir}/validation_summary.csv",
        index=False
    )

    df.to_csv(
        f"{output_dir}/healthcare_dataset_cleaned.csv",
        index=False
    )

    return df

if __name__ == "__main__":
    validate_and_standardize("healthcare_dataset.csv")
